In [1]:
import pandas as pd
import numpy as np
import os

print("Starting Venue Trend Analysis...")

# 1. Load Data
matches = pd.read_csv('../data/raw/matches.csv')
deliveries = pd.read_csv('../data/raw/deliveries.csv')

# 2. Clean Venue Names (Standardizing to avoid duplicates like 'M Chinnaswamy' vs 'M.Chinnaswamy')
venue_mapping = {
    'M.Chinnaswamy Stadium': 'M Chinnaswamy Stadium',
    'Punjab Cricket Association IS Bindra Stadium, Mohali': 'Punjab Cricket Association Stadium, Mohali',
    'Rajiv Gandhi International Stadium, Uppal': 'Rajiv Gandhi International Stadium',
    'Wankhede Stadium, Mumbai': 'Wankhede Stadium',
    'MA Chidambaram Stadium, Chepauk': 'MA Chidambaram Stadium',
    'MA Chidambaram Stadium, Chepauk, Chennai': 'MA Chidambaram Stadium'
}
matches['venue'] = matches['venue'].replace(venue_mapping)

# 3. Calculate Toss Advantages & Win Rates per Venue
# Determine if the team batting first won (1) or lost (0)
matches['bat_first_won'] = np.where(
    ((matches['toss_decision'] == 'bat') & (matches['toss_winner'] == matches['winner'])) |
    ((matches['toss_decision'] == 'field') & (matches['toss_winner'] != matches['winner'])), 
    1, 0
)

venue_stats = matches.groupby('venue').agg(
    total_matches=('id', 'count'),
    toss_bat_first_freq=('toss_decision', lambda x: (x == 'bat').mean()),
    bat_first_win_pct=('bat_first_won', 'mean')
).reset_index()

# 4. Calculate Average 1st Innings Score per Venue
# Filter for 1st innings only, sum runs per match, then average by venue
first_innings = deliveries[deliveries['inning'] == 1]
match_scores = first_innings.groupby(['match_id']).agg(total_runs=('total_runs', 'sum')).reset_index()

# Merge with matches to get the venue for each match
match_scores = match_scores.merge(matches[['id', 'venue']], left_on='match_id', right_on='id')
avg_scores = match_scores.groupby('venue').agg(avg_1st_innings_score=('total_runs', 'mean')).reset_index()

# 5. Combine and Save Final Venue Intelligence Data
final_venue_intel = venue_stats.merge(avg_scores, on='venue')

# Filter out venues with less than 5 matches to remove noise/outliers
final_venue_intel = final_venue_intel[final_venue_intel['total_matches'] >= 5]

# Format decimals for clean UI display later
final_venue_intel['toss_bat_first_freq'] = (final_venue_intel['toss_bat_first_freq'] * 100).round(1)
final_venue_intel['bat_first_win_pct'] = (final_venue_intel['bat_first_win_pct'] * 100).round(1)
final_venue_intel['avg_1st_innings_score'] = final_venue_intel['avg_1st_innings_score'].round(0).astype(int)

# Ensure processed folder exists and save
os.makedirs('../data/processed', exist_ok=True)
final_venue_intel.to_csv('../data/processed/venue_intelligence.csv', index=False)

print("✅ Venue analysis complete! Data saved to 'data/processed/venue_intelligence.csv'")
display(final_venue_intel.sort_values('total_matches', ascending=False).head(10))

Starting Venue Trend Analysis...
✅ Venue analysis complete! Data saved to 'data/processed/venue_intelligence.csv'


,venue,total_matches,toss_bat_first_freq,bat_first_win_pct,avg_1st_innings_score
50,Wankhede Stadium,118,25.4,45.8,170
25,MA Chidambaram Stadium,85,56.5,57.6,164
23,M Chinnaswamy Stadium,80,11.2,45.0,169
14,Eden Gardens,77,36.4,39.0,160
37,Rajiv Gandhi International Stadium,64,43.8,42.2,158
16,Feroz Shah Kotla,60,43.3,45.0,162
41,Sawai Mansingh Stadium,47,40.4,31.9,158
36,"Punjab Cricket Association Stadium, Mohali",46,34.8,43.5,163
13,Dubai International Cricket Stadium,46,41.3,50.0,164
45,Sheikh Zayed Stadium,29,51.7,44.8,159
